# 03 - Long document, async + chunked

Demonstrates **chunked map-reduce extraction** and the speedup from **async concurrency**. Runs offline using `MockProvider` with an injected latency so the timing is realistic without burning API calls.

What you will see:
1. A document long enough to trigger automatic chunking.
2. Sync compression — chunks processed one at a time.
3. Async compression — chunks processed concurrently via `asyncio.gather`.
4. Wall-clock comparison.

When this matters in real life: your provider has per-request latency of 1-3 s. A 20-chunk document takes 20-60 s sync; 4-8 s async at `concurrency=8`.


## Setup

In [ ]:
import asyncio
import json
import time

from narrato import Compressor
from narrato.providers.mock import MockProvider


## A long source document

Synthetic but realistic — repeated paragraphs of meeting-style content, ~25 KB. Real Q3 strategy meetings often produce this much text per session.


In [ ]:
PARAGRAPH = (
    "Meeting note: the team reviewed the Q3 progress on Project Halibut. Engineering is two "
    "weeks behind because of unexpected complexity in the new ranking algorithm. Design has "
    "shipped onboarding screens that lifted day-7 retention by 4 percentage points. Data "
    "identified the sign-up to first-message drop-off as the largest leak in the funnel. "
    "Product proposed three priorities: finish Halibut by November 30, extend onboarding to "
    "all new users by November 15, and ship the AI assistant for paid users by December 20. "
    "Engineering needs one more backend hire to hit November 30. Recruiting will be consulted. "
)

SOURCE = "\n\n".join(
    f"Section {i+1}.\n" + PARAGRAPH * 3
    for i in range(20)
)
print(f"source length: {len(SOURCE):,} chars ({len(SOURCE)/4:,.0f} approx tokens)")


## A mock provider with injected latency

Each `complete_json` call sleeps for `delay` seconds — simulates a real LLM round-trip. The async version sleeps the same amount but yields the event loop, so concurrent calls overlap.


In [ ]:
PAYLOAD = {
    "summary": "Section talks about Q3 progress on Project Halibut.",
    "entities": ["Project Halibut", "Engineering", "Design", "Data", "Recruiting"],
    "dates": ["November 30", "November 15", "December 20"],
    "claims": [
        "Halibut is two weeks behind.",
        "Onboarding lifted day-7 retention by 4 points.",
        "Sign-up to first-message is the biggest funnel leak.",
    ],
}

DELAY_PER_CALL = 0.5   # seconds; tweak up to feel real-world latency

def make_latency_mock(delay: float) -> MockProvider:
    """MockProvider where every call (sync or async) sleeps `delay` seconds.

    Both wrappers route to the same captured `real_complete_json` — the async
    wrapper deliberately bypasses `acomplete_json` so the async sleep is the
    only blocking time, letting `asyncio.gather` overlap concurrent calls.
    """
    base = MockProvider(payload=PAYLOAD)
    real_complete_json = base.complete_json   # captured before patching

    def sync_wrapper(system, user, model, schema=None, max_tokens=2048, temperature=0.0):
        time.sleep(delay)
        return real_complete_json(system, user, model, schema, max_tokens, temperature)

    async def async_wrapper(system, user, model, schema=None, max_tokens=2048, temperature=0.0):
        await asyncio.sleep(delay)
        return real_complete_json(system, user, model, schema, max_tokens, temperature)

    base.complete_json = sync_wrapper       # type: ignore[assignment]
    base.acomplete_json = async_wrapper     # type: ignore[assignment]
    return base


## Sync run — chunks processed one at a time


In [ ]:
sync_provider = make_latency_mock(DELAY_PER_CALL)

c_sync = Compressor(
    provider=sync_provider,
    source_lang="en",
    layers=["extract"],
    schema="qa",
    chunk_chars=2000,    # forces several chunks on the long doc
    overlap_chars=100,
)

t0 = time.perf_counter()
sync_result = c_sync.compress(SOURCE)
sync_elapsed = time.perf_counter() - t0

print(f"sync   : {sync_elapsed:.2f} s, {sync_result.stats['extract']['chunks']} chunks, {sync_provider.calls_complete_json} llm calls")


## Async run — chunks processed concurrently


In [ ]:
async_provider = make_latency_mock(DELAY_PER_CALL)

c_async = Compressor(
    provider=async_provider,
    source_lang="en",
    layers=["extract"],
    schema="qa",
    chunk_chars=2000,
    overlap_chars=100,
)

async def go():
    return await c_async.acompress(SOURCE, concurrency=8)

t0 = time.perf_counter()
async_result = asyncio.run(go())
async_elapsed = time.perf_counter() - t0

print(f"async  : {async_elapsed:.2f} s, {async_result.stats['extract']['chunks']} chunks, {async_provider.calls_complete_json} llm calls, concurrency={async_result.stats['extract']['concurrency']}")


## Side-by-side


In [ ]:
speedup = sync_elapsed / async_elapsed if async_elapsed else 1.0

print(f"{'mode':<8} {'wall-clock':>12} {'chunks':>8} {'calls':>8}")
print("-" * 42)
print(f"{'sync':<8} {sync_elapsed:>11.2f}s {sync_result.stats['extract']['chunks']:>8} {sync_provider.calls_complete_json:>8}")
print(f"{'async':<8} {async_elapsed:>11.2f}s {async_result.stats['extract']['chunks']:>8} {async_provider.calls_complete_json:>8}")
print()
print(f"speedup: {speedup:.2f}x")


## Did the merge work?

Both modes should produce the *same* merged payload. Chunks return identical canned data, dedupe should leave one of each entity / date / claim.


In [ ]:
sync_payload = json.loads(sync_result.payload)
async_payload = json.loads(async_result.payload)

print("sync entities :", sync_payload["entities"])
print("async entities:", async_payload["entities"])
print()
print("equal:", sync_payload == async_payload)


## Practical guidance

- Use sync `compress()` for short, single-shot calls. Simpler, no event loop.
- Use async `acompress()` whenever you have many chunks or many documents.
- `concurrency` should be lower than your provider's per-key parallel-request limit. Default `4` is safe for most paid tiers.
- Speedup is bounded by `min(chunks, concurrency)` and by per-call latency.
